# Customer Churn, SHAP Explainability

ROC-AUC tells you how well the model discriminates, but it doesn't tell you why it makes individual predictions. SHAP values decompose each prediction into per-feature contributions, which is useful both for debugging the model and for translating results into something actionable for a business team.

In [1]:
import pandas as pd
import numpy as np
import shap
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
os.makedirs("../visuals", exist_ok=True)

In [2]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)
df.drop(columns=["customerID"], inplace=True)
df["Churn"] = (df["Churn"] == "Yes").astype(int)

le = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

X = df.drop(columns=["Churn"])
y = df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    base_score=0.5,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric="logloss", random_state=42, use_label_encoder=False
)
model.fit(X_train, y_train, verbose=False)
print("Model ready.")

Model ready.


In [3]:
import xgboost as xgb
dtest = xgb.DMatrix(X_test, feature_names=X_test.columns.tolist())
contribs = model.get_booster().predict(dtest, pred_contribs=True)
shap_values = contribs[:, :-1]
expected_value = float(contribs[0, -1])
print(f"SHAP values shape: {shap_values.shape}")

SHAP values shape: (1409, 19)


The summary plot ranks features by mean absolute SHAP value (overall importance) and shows the direction of each feature's effect. Tenure at the top is expected from EDA. The color gradient on each feature shows whether high values push the prediction up or down.

In [4]:
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP Summary Plot", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../visuals/shap_summary.png", bbox_inches="tight")
plt.show()

A waterfall plot for a single prediction shows which features pushed this customer's churn probability above the baseline. Useful for explaining an individual case to a retention team.

In [5]:
churn_idx = X_test[y_test == 1].index[0]
sample_pos = X_test.index.get_loc(churn_idx)

explanation = shap.Explanation(
    values=shap_values[sample_pos],
    base_values=expected_value,
    data=X_test.iloc[sample_pos].values,
    feature_names=X_test.columns.tolist()
)

fig, ax = plt.subplots(figsize=(10, 6))
shap.waterfall_plot(explanation, max_display=12, show=False)
plt.title("SHAP Waterfall: Single Prediction", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("../visuals/shap_waterfall.png", bbox_inches="tight")
plt.show()

Dependence plots for the top two features show how the SHAP value changes across the feature's range, and whether a second feature (color) interacts with it.

In [6]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top2_features = X_test.columns[np.argsort(mean_abs_shap)[::-1][:2]].tolist()
print(f"Top 2 features: {top2_features}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, feat in enumerate(top2_features):
    shap.dependence_plot(feat, shap_values, X_test, ax=axes[i], show=False)
    axes[i].set_title(f"SHAP Dependence: {feat}", fontweight="bold")

plt.tight_layout()
plt.savefig("../visuals/shap_dependence.png", bbox_inches="tight")
plt.show()

Top 2 features: ['Contract', 'tenure']


In [7]:
importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "shap_mean": mean_abs_shap
}).sort_values("shap_mean", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df["feature"][::-1], importance_df["shap_mean"][::-1], color="#4C72B0")
ax.set_xlabel("Mean |SHAP Value|")
ax.set_title("Top 15 Features by Mean |SHAP Value|", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../visuals/shap_importance_bar.png", bbox_inches="tight")
plt.show()